In [ ]:
#Silver Price Prediction - Full ML Pipeline
#EDA -> Preprocessing -> Feature Engineering -> Modeling -> Evaluation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# สำรวจข้อมูล (EDA)
data = pd.read_csv("../../Data/CSV/silver_prices_10years.csv")

print("=== DATA HEAD ===")
print(data.head())

print("\n=== DATA INFO ===")
print(data.info())

print("\n=== MISSING VALUES ===")
print(data.isnull().sum())

# แปลงวันที่
data['Date'] = pd.to_datetime(data['Date'])
data = data.sort_values('Date')

# ดูกราฟราคาปิด
plt.figure(figsize=(12,5))
plt.plot(data['Date'], data['Close'])
plt.title("Silver Close Price Over Time")
plt.show()


# เตรียมข้อมูล (Preprocessing)
# เลือกคอลัมน์ที่ใช้
data = data[['Date','Close','High','Low','Open','Volume']]
#Previous Day Price

df['Prev_Close'] = df['Close'].shift(1)

# 7-Day Moving Average
df['MA7'] = df['Close'].rolling(window=7).mean()

# 30-Day Moving Average
df['MA30'] = df['Close'].rolling(window=30).mean()

##lag feature 
df['Lag1'] = df['Close'].shift(1)
df['Lag2'] = df['Close'].shift(2)
df['Lag3'] = df['Close'].shift(3)
df['Lag5'] = df['Close'].shift(5)
df['Lag10'] = df['Close'].shift(10)

# for  more acc
# momantum
df['Return'] = df['Close'].pct_change()
##short term mm
df['MA3'] = df['Close'].rolling(3).mean()
#volaitixxx
df['Volatility'] = df['Close'].rolling(7).std()
df['Return'] = df['Close'].pct_change()
df['Volatility'] = df['Close'].rolling(7).std()

## Drop NA
# Feature Engineering
df['Prev_Close'] = df['Close'].shift(1)
df['MA7'] = df['Close'].rolling(7).mean()
df['MA30'] = df['Close'].rolling(30).mean()

df['Return'] = df['Close'].pct_change()
df['Volatility'] = df['Close'].rolling(7).std()

# Target
df['Tomorrow_Close'] = df['Close'].shift(-1)

# ลบ missing (ถ้ามี)
data = data.dropna()


# Feature Engineering
# สร้าง Moving Average
data['MA10'] = data['Close'].rolling(window=10).mean()
data['MA30'] = data['Close'].rolling(window=30).mean()

# ลบ NaN จาก rolling
data = data.dropna()

# เลือก feature
features = ['Close','High','Low','Open','Volume','MA10','MA30']

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data[features])


#สร้าง Time Series Dataset (60 วันย้อนหลัง)
sequence_length = 60
X = []
y = []

for i in range(sequence_length, len(scaled_data)):
    X.append(scaled_data[i-sequence_length:i])
    y.append(scaled_data[i][0])  # ทำนาย Close

X = np.array(X)
y = np.array(y)

# แบ่ง Train/Test
train_size = int(len(X)*0.8)

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]


# สร้างโมเดล LSTM
model = Sequential()
model.add(LSTM(64, return_sequences=True, input_shape=(X.shape[1], X.shape[2])))
model.add(Dropout(0.2))

model.add(LSTM(64))
model.add(Dropout(0.2))

model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')

model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)


# ทำนาย
predictions = model.predict(X_test)

# แปลงกลับค่า Close จริง
close_index = features.index('Close')

# สร้าง array เต็มเพื่อ inverse
temp_pred = np.zeros((len(predictions), len(features)))
temp_pred[:, close_index] = predictions[:,0]

temp_true = np.zeros((len(y_test), len(features)))
temp_true[:, close_index] = y_test

predictions_actual = scaler.inverse_transform(temp_pred)[:,close_index]
y_test_actual = scaler.inverse_transform(temp_true)[:,close_index]

#ประเมินผลโมเดล
mse = mean_squared_error(y_test_actual, predictions_actual)
mae = mean_absolute_error(y_test_actual, predictions_actual)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_actual, predictions_actual)

# ตารางสรุปผล
results = pd.DataFrame({
    "Metric": ["R2 Score","MSE","MAE","RMSE"],
    "Value": [r2, mse, mae, rmse]
})
print("\n=== MODEL PERFORMANCE ===")
print(results)

#กราฟเปรียบเทียบราคา
plt.figure(figsize=(12,6))
plt.plot(y_test_actual, label="Actual")
plt.plot(predictions_actual, label="Predicted")
plt.legend()
plt.title("Actual vs Predicted Silver Price")
plt.show()

#กราฟเปรียบเทียบ Metrics
plt.figure(figsize=(8,5))
sns.barplot(x="Metric", y="Value", data=results)
plt.title("Model Evaluation Metrics")
plt.show()


ModuleNotFoundError: No module named 'tensorflow'

In [6]:
import tensorflow as tf
print(tf.__version__)


ModuleNotFoundError: No module named 'tensorflow'